# Fund Flows and Past Performance: Evidence from Active U.S. Domestic Equity Funds

**Bachelor Thesis — Empirical Finance**  

## Overview

This notebook performs a CRSP-based empirical analysis of the relationship between
past mutual fund performance and subsequent investor flows, for active U.S. domestic
equity funds over the period 1999–2011.

**Replication target:** The flow-performance analysis in the style of
Barber, Huang, Odean & Schwarz (2022), restricted to the active equity (EDC) fund universe.

**Portfolio identifier:** WFICN from MFLinks is used as the fund-level identifier
throughout, because `crsp_portno` coverage is sparse before 2003.

**Key research question:** Do funds with higher past returns attract disproportionately
higher investor inflows? Is the flow-performance relationship convex (nonlinear)?

---

### Notebook structure
1. Imports and path setup
2. Load raw CRSP data
3. WFICN mapping and share-class aggregation → Step 1 dataset
4. Fund Summary merge, EDC filter, flow calculation → Step 2 dataset
5. QC cleaning and past performance construction → Step 3 (regression-ready) dataset
6. Main 12-month flow-performance regressions
7. 12-month quintile flow-performance analysis
8. 6-month robustness check
9. Final overview and QC

## 1. Imports and Path Setup

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = '/Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data'
RAW  = BASE + '/real, clear data/raw/new data/'
PROC = BASE + '/NEW_BA_analysis/data/processed/'
TABS = BASE + '/NEW_BA_analysis/output/tables/'

Path(PROC).mkdir(parents=True, exist_ok=True)
Path(TABS).mkdir(parents=True, exist_ok=True)

print('Paths configured.')
print(f'  RAW  : {RAW}')
print(f'  PROC : {PROC}')
print(f'  TABS : {TABS}')

Paths configured.
  RAW  : /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data/real, clear data/raw/new data/
  PROC : /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data/NEW_BA_analysis/data/processed/
  TABS : /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data/NEW_BA_analysis/output/tables/


### Shared helpers (used across sections)

In [2]:
def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def tbl_to_md(df):
    cols  = df.columns.tolist()
    lines = ['| ' + ' | '.join(str(c) for c in cols) + ' |',
             '| ' + ' | '.join(['---'] * len(cols)) + ' |']
    for _, row in df.iterrows():
        lines.append('| ' + ' | '.join(str(v) for v in row) + ' |')
    return '\n'.join(lines)

def run_ols(Y, X, cluster_ids, label=''):
    m = sm.OLS(Y, X).fit(cov_type='cluster', cov_kwds={'groups': cluster_ids})
    if label:
        print(f'  {label}  N={int(m.nobs):,}  R²={m.rsquared:.4f}  adj-R²={m.rsquared_adj:.4f}')
    return m

print('Helpers defined.')

Helpers defined.


## 2. Load Raw CRSP Data

Three source files from CRSP / MFLinks:
- `monthly_returns_1991_2011.csv` — share-class monthly returns and TNA
- `fund_summary_quarterly_1991_2011.csv` — quarterly fund characteristics
- `mflinks_crsp_fundno_wficn.csv` — crosswalk from `crsp_fundno` to WFICN portfolio identifier

In [3]:
print('Loading raw files …')
ret = pd.read_csv(RAW + 'monthly_returns_1991_2011.csv',        low_memory=False)
fsq = pd.read_csv(RAW + 'fund_summary_quarterly_1991_2011.csv', low_memory=False)
mfl = pd.read_csv(RAW + 'mflinks_crsp_fundno_wficn.csv',       low_memory=False)

for df in (ret, fsq, mfl):
    df.columns = df.columns.str.lower().str.strip()

ret['caldt'] = pd.to_datetime(ret['caldt'])
fsq['caldt'] = pd.to_datetime(fsq['caldt'])

# 'R' codes in mret indicate reinvestment-only months; coerce to NaN
ret['mret'] = pd.to_numeric(ret['mret'], errors='coerce')
ret['mtna'] = pd.to_numeric(ret['mtna'], errors='coerce')
ret['year'] = ret['caldt'].dt.year

print(f'  monthly_returns   : {len(ret):>10,} rows | {ret["crsp_fundno"].nunique():,} fundnos'
      f' | {ret["caldt"].min().date()} – {ret["caldt"].max().date()}')
print(f'  fund_summary_qtr  : {len(fsq):>10,} rows | {fsq["crsp_fundno"].nunique():,} fundnos'
      f' | {fsq["caldt"].min().date()} – {fsq["caldt"].max().date()}')
print(f'  mflinks           : {len(mfl):>10,} rows | {mfl["crsp_fundno"].nunique():,} fundnos'
      f' | {mfl["wficn"].nunique():,} WFICNs')

Loading raw files …


  monthly_returns   :  4,526,907 rows | 48,765 fundnos | 1991-01-31 – 2011-12-30
  fund_summary_qtr  :  1,104,735 rows | 45,495 fundnos | 1999-03-31 – 2011-12-30
  mflinks           :     50,380 rows | 49,975 fundnos | 15,560 WFICNs


## 3. WFICN Mapping and Share-Class Aggregation

**Why WFICN instead of `crsp_portno`?**  
The CRSP `fund_portfolio_map` assigns `crsp_portno` starting mainly from 2003,
leaving ~65% of observations unmapped for the 1991–2002 window.
MFLinks (`mflinks_crsp_fundno_wficn.csv`) provides a static `crsp_fundno → WFICN`
crosswalk that achieves 93–99% coverage throughout 1991–2011.

**Share-class aggregation:**  
Each WFICN may span multiple share classes (`crsp_fundno`). We aggregate to
the WFICN × month level using TNA-weighted returns:

$$\text{portfolio\_return}_t = \frac{\sum_i r_{i,t} \cdot \text{TNA}_{i,t}}{\sum_i \text{TNA}_{i,t}}$$

Portfolio TNA = sum of all share-class TNAs. Rows with missing TNA are excluded
from the return average (but included in portfolio_tna via `sum`).

In [4]:
# ── Canonical WFICN: modal assignment for fundnos with multiple WFICNs ──────
wficn_canonical = (
    mfl.groupby('crsp_fundno')['wficn']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)

mapped   = ret.merge(wficn_canonical, on='crsp_fundno', how='inner')
unmapped = ret[~ret['crsp_fundno'].isin(wficn_canonical['crsp_fundno'])]
pct_map  = 100 * len(mapped) / len(ret)

print(f'  Total observations : {len(ret):>10,}')
print(f'  Mapped (have WFICN): {len(mapped):>10,}  ({pct_map:.1f}%)')
print(f'  Unmapped           : {len(unmapped):>10,}  ({100-pct_map:.1f}%)')
print(f'  Unique WFICNs      : {mapped["wficn"].nunique():>9,}')

  Total observations :  4,526,907
  Mapped (have WFICN):  3,352,414  (74.1%)
  Unmapped           :  1,174,493  (25.9%)
  Unique WFICNs      :    10,771


In [5]:
# ── TNA-weighted return aggregation ──────────────────────────────────────────
mapped_clean = mapped.dropna(subset=['mtna']).copy()

valid_for_return = mapped_clean['mret'].notna() & (mapped_clean['mtna'] > 0)
mapped_clean['ret_x_tna']   = np.where(valid_for_return,
                                         mapped_clean['mret'] * mapped_clean['mtna'],
                                         np.nan)
mapped_clean['tna_for_ret'] = np.where(valid_for_return, mapped_clean['mtna'], np.nan)
mapped_clean['valid_sc']    = valid_for_return.astype(int)

grp = mapped_clean.groupby(['wficn', 'caldt'])
agg = pd.DataFrame({
    'portfolio_tna'                    : grp['mtna'].sum(),
    'ret_num'                          : grp['ret_x_tna'].sum(min_count=1),
    'ret_den'                          : grp['tna_for_ret'].sum(min_count=1),
    'n_share_classes'                  : grp['crsp_fundno'].count(),
    'n_valid_share_classes_for_return' : grp['valid_sc'].sum(),
}).reset_index()

agg['portfolio_return'] = agg['ret_num'] / agg['ret_den']
agg = agg.drop(columns=['ret_num', 'ret_den'])

print(f'  Step 1 dataset: {len(agg):,} WFICN-month obs | {agg["wficn"].nunique():,} WFICNs'
      f' | {agg["caldt"].min().date()} – {agg["caldt"].max().date()}')
print(f'  Avg share classes / WFICN-month : {agg["n_share_classes"].mean():.2f}')
print(f'  WFICN-months with missing return: {agg["portfolio_return"].isna().sum():,}')

# Save Step 1 dataset
agg.to_csv(PROC + 'final_step1_wficn_portfolio_month.csv', index=False)
print(f'\n  Saved → final_step1_wficn_portfolio_month.csv')

  Step 1 dataset: 1,241,561 WFICN-month obs | 10,740 WFICNs | 1991-01-31 – 2011-12-30
  Avg share classes / WFICN-month : 2.53
  WFICN-months with missing return: 78,487



  Saved → final_step1_wficn_portfolio_month.csv


## 4. Fund Summary Merge, EDC Filter, ETF/Index Exclusions, and Flow Calculation

### 4.1 Strategy

We attach quarterly fund characteristics from `fund_summary_quarterly` to each
WFICN-month using an **as-of (backward-looking) merge** — for each monthly
observation we use the most recent quarterly snapshot with date ≤ current month.
This avoids look-ahead bias.

### 4.2 Sample filters (applied sequentially)

1. Drop observations with missing or zero portfolio TNA
2. Drop observations with missing portfolio return
3. Keep sample window: 1997-07 to 2011-12
4. **Keep EDC funds**: `crsp_obj_cd` starts with `'EDC'` (small/mid-cap domestic equity)
5. Exclude ETFs (flag-based + name-based)
6. Exclude index funds (flag-based + name-based)
7. Exclude residual non-target types (international, bond, money market, etc.)

### 4.3 Flow formula

$$\text{flow}_t = \frac{\text{TNA}_t}{\text{TNA}_{t-1}} - (1 + r_t)$$

Only valid when the calendar gap to the previous observation is exactly
one month (28–35 days). Flow is winsorized at the 1st/99th percentiles.

In [6]:
# ── Name-based exclusion keyword lists ───────────────────────────────────────
INDEX_TERMS   = ['index', ' idx ', 's&p 500', 's&p500', 'russell', 'wilshire',
                 'msci', 'dow jones', 'nasdaq composite', 'nasdaq-100',
                 'stoxx', 'ftse', ' ixus']
ETF_TERMS     = ['etf', 'exchange traded fund', 'exchange-traded fund', 'etn ']
NONEQUITY_TERMS = [
    'international', 'global', 'world', 'emerging market', 'foreign', 'overseas',
    'europe', 'european', 'pacific', 'asia', 'japan', 'china', 'india', 'latin america',
    'bond', 'fixed income', 'treasury', 'government', 'municipal', 'corporate debt',
    'high yield', 'credit', 'income fund', 'money market', 'cash management',
    'balanced', 'allocation', 'flexible portfolio', 'hybrid',
    'real estate', 'reit', 'sector fund', 'target date', 'retirement fund',
    'commodity', 'precious metal', 'gold fund',
]

# ── Load Step 1 output ────────────────────────────────────────────────────────
step1 = pd.read_csv(PROC + 'final_step1_wficn_portfolio_month.csv', low_memory=False)
step1['caldt'] = pd.to_datetime(step1['caldt'])

# ── Aggregate FSQ to WFICN-quarter ───────────────────────────────────────────
fsq_w = fsq.merge(wficn_canonical, on='crsp_fundno', how='inner')
fsq_w['tna_latest']    = pd.to_numeric(fsq_w['tna_latest'], errors='coerce')
fsq_w['first_offer_dt']= pd.to_datetime(fsq_w['first_offer_dt'], errors='coerce')
for c in ['exp_ratio','turn_ratio','mgmt_fee','actual_12b1']:
    fsq_w[c] = pd.to_numeric(fsq_w[c], errors='coerce')

# Representative row: largest-TNA share class per WFICN-quarter
cat_cols = ['fund_name','ticker','crsp_obj_cd','lipper_asset_cd',
            'lipper_obj_cd','lipper_obj_name','lipper_class','lipper_class_name',
            'policy','delist_cd']
fsq_rep = (
    fsq_w
    .sort_values(['wficn','caldt','tna_latest'], ascending=[True,True,False], na_position='last')
    .drop_duplicates(subset=['wficn','caldt'])
    [['wficn','caldt'] + cat_cols]
)

# Conservative flags: ANY share class flagged → whole WFICN-quarter is flagged
fsq_w['_idx'] = fsq_w['index_fund_flag'].notna().astype(np.int8)
fsq_w['_etf'] = fsq_w['et_flag'].notna().astype(np.int8)
fsq_w['_ded'] = (fsq_w['dead_flag'].astype(str).str.upper() == 'Y').astype(np.int8)
flag_agg = (
    fsq_w.groupby(['wficn','caldt'])
    .agg(has_index_flag=('_idx','max'), has_et_flag=('_etf','max'),
         dead_flag_any =('_ded','max'))
    .reset_index()
)
flag_agg['has_index_flag'] = flag_agg['has_index_flag'].astype(bool)
flag_agg['has_et_flag']    = flag_agg['has_et_flag'].astype(bool)

# TNA-weighted numeric characteristics
def tna_wavg(df, col):
    valid = df[col].notna() & df['tna_latest'].notna() & (df['tna_latest'] > 0)
    df = df.copy()
    df['_n'] = np.where(valid, df[col] * df['tna_latest'], np.nan)
    df['_d'] = np.where(valid, df['tna_latest'], np.nan)
    g = df.groupby(['wficn','caldt'])
    r = (g['_n'].sum(min_count=1) / g['_d'].sum(min_count=1)).reset_index()
    r.columns = ['wficn','caldt', col]
    return r

num_cols = ['exp_ratio','turn_ratio','mgmt_fee','actual_12b1']
num_agg  = tna_wavg(fsq_w, num_cols[0])
for c in num_cols[1:]:
    num_agg = num_agg.merge(tna_wavg(fsq_w, c), on=['wficn','caldt'], how='outer')

first_offer = fsq_w.groupby('wficn')['first_offer_dt'].min().reset_index()
fsq_agg = (
    fsq_rep
    .merge(flag_agg,    on=['wficn','caldt'], how='left')
    .merge(num_agg,     on=['wficn','caldt'], how='left')
    .merge(first_offer, on='wficn',           how='left')
)
print(f'  FSQ WFICN-quarter: {len(fsq_agg):,} rows | {fsq_agg["wficn"].nunique():,} WFICNs'
      f' | {fsq_agg["caldt"].min().date()} – {fsq_agg["caldt"].max().date()}')

  FSQ WFICN-quarter: 306,590 rows | 10,319 WFICNs | 1999-03-31 – 2011-12-30


In [7]:
# ── As-of merge: monthly ← quarterly (backward, no tolerance) ───────────────
# merge_asof requires global sort on the 'on' column (not per-group).
step1_s = step1.sort_values('caldt').reset_index(drop=True)
fsq_s   = fsq_agg.sort_values('caldt').reset_index(drop=True)

merged = pd.merge_asof(
    step1_s,
    fsq_s.rename(columns={'caldt': 'fsq_caldt'}),
    left_on='caldt', right_on='fsq_caldt',
    by='wficn', direction='backward',
)
merged = merged.sort_values(['wficn','caldt']).reset_index(drop=True)

pct_matched = merged['crsp_obj_cd'].notna().mean() * 100
print(f'  After as-of merge: {len(merged):,} rows | {pct_matched:.1f}% have crsp_obj_cd')

  After as-of merge: 1,241,561 rows | 67.6% have crsp_obj_cd


In [8]:
# ── Sequential sample filters ────────────────────────────────────────────────
construction = []

def drop_step(df, mask, label, note=''):
    n0, w0 = len(df), df['wficn'].nunique()
    out = df[mask].copy()
    rm_n = n0 - len(out); rm_w = w0 - out['wficn'].nunique()
    construction.append({'Step': label, 'Obs': len(out), 'WFICN': out['wficn'].nunique(),
                         'Removed obs': rm_n, 'Note': note})
    return out

df2 = merged.copy()
construction.append({'Step': '0. After merge', 'Obs': len(df2),
                     'WFICN': df2['wficn'].nunique(), 'Removed obs': 0, 'Note': ''})

df2 = drop_step(df2, df2['portfolio_tna'].notna() & (df2['portfolio_tna'] > 0),
                '1. Require positive TNA')
df2 = drop_step(df2, df2['portfolio_return'].notna(),
                '2. Require non-null return')
df2 = drop_step(df2, df2['caldt'].between('1997-07-01', '2011-12-31'),
                '3. Sample window 1997–2011')

# EDC filter: crsp_obj_cd must start with 'EDC' (not just 'ED', which includes
# growth/blend/income sub-categories outside our active core-equity scope)
df2 = drop_step(df2, df2['crsp_obj_cd'].str.startswith('EDC', na=False),
                "4. Keep EDC (crsp_obj_cd startswith 'EDC')")

# ETF exclusions: flag-based takes priority; name-based catches remainder
etf_flag = df2['has_et_flag'].fillna(False)
txt = (df2['fund_name'].fillna('').str.lower() + ' ' +
       df2['lipper_obj_name'].fillna('').str.lower())
etf_name = txt.apply(lambda t: any(k in t for k in ETF_TERMS))
df2 = drop_step(df2, ~(etf_flag | etf_name), '5. Exclude ETFs')

idx_flag = df2['has_index_flag'].fillna(False)
txt2 = (df2['fund_name'].fillna('').str.lower() + ' ' +
        df2['lipper_obj_name'].fillna('').str.lower())
idx_name = txt2.apply(lambda t: any(k in t for k in INDEX_TERMS))
df2 = drop_step(df2, ~(idx_flag | idx_name), '6. Exclude index funds')

txt3 = (df2['fund_name'].fillna('').str.lower() + ' ' +
        df2['lipper_obj_name'].fillna('').str.lower() + ' ' +
        df2['lipper_class_name'].fillna('').str.lower() + ' ' +
        df2['policy'].fillna('').str.lower())
nontarget = txt3.apply(lambda t: any(k in t for k in NONEQUITY_TERMS))
df2 = drop_step(df2, ~nontarget, '7. Exclude residual non-target types')

print('Sample construction:')
for row in construction:
    print(f"  {row['Step']:<42}  obs={row['Obs']:>8,}  WFICN={row['WFICN']:>5,}"
          f"  (−{row['Removed obs']:,})" if row['Removed obs'] else
          f"  {row['Step']:<42}  obs={row['Obs']:>8,}  WFICN={row['WFICN']:>5,}")

Sample construction:
  0. After merge                              obs=1,241,561  WFICN=10,740
  1. Require positive TNA                     obs=1,241,356  WFICN=10,740  (−205)
  2. Require non-null return                  obs=1,163,074  WFICN=10,621  (−78,282)
  3. Sample window 1997–2011                  obs= 971,408  WFICN=10,325  (−191,666)
  4. Keep EDC (crsp_obj_cd startswith 'EDC')  obs= 135,654  WFICN=1,653  (−835,754)
  5. Exclude ETFs                             obs= 129,782  WFICN=1,565  (−5,872)
  6. Exclude index funds                      obs= 110,458  WFICN=1,340  (−19,324)
  7. Exclude residual non-target types        obs= 108,310  WFICN=1,322  (−2,148)


In [9]:
# ── Flow calculation ─────────────────────────────────────────────────────────
df2 = df2.sort_values(['wficn','caldt']).reset_index(drop=True)

df2['lag_portfolio_tna'] = df2.groupby('wficn')['portfolio_tna'].shift(1)
df2['lag_caldt']         = df2.groupby('wficn')['caldt'].shift(1)
df2['caldt_gap_days']    = (df2['caldt'] - df2['lag_caldt']).dt.days

# Only compute flow when the gap to the previous record is exactly one month
valid_lag = (
    df2['lag_portfolio_tna'].notna() &
    (df2['lag_portfolio_tna'] > 0) &
    (df2['portfolio_tna'] > 0) &
    df2['caldt_gap_days'].between(28, 35)
)
df2['flow'] = np.where(
    valid_lag,
    df2['portfolio_tna'] / df2['lag_portfolio_tna'] - (1 + df2['portfolio_return']),
    np.nan
)

p01 = df2['flow'].quantile(0.01)
p99 = df2['flow'].quantile(0.99)
df2['flow_winsorized'] = df2['flow'].clip(lower=p01, upper=p99)

df2['fund_age_years'] = np.where(
    df2['first_offer_dt'].notna(),
    (df2['caldt'] - pd.to_datetime(df2['first_offer_dt'])).dt.days / 365.25,
    np.nan
)

print(f'  Final EDC sample: {len(df2):,} obs | {df2["wficn"].nunique():,} WFICNs'
      f' | {df2["caldt"].min().date()} – {df2["caldt"].max().date()}')
print(f'  Flow winsorization bounds: [{p01:.4f}, {p99:.4f}]')
print(f'  Obs with valid flow: {df2["flow"].notna().sum():,} ({df2["flow"].notna().mean()*100:.1f}%)')

# Save Step 2 dataset
out_cols = ['wficn','caldt','portfolio_tna','lag_portfolio_tna','portfolio_return',
            'flow','flow_winsorized','n_share_classes','n_valid_share_classes_for_return',
            'fund_name','ticker','crsp_obj_cd','lipper_asset_cd','lipper_obj_cd',
            'lipper_obj_name','lipper_class','lipper_class_name','policy',
            'has_index_flag','has_et_flag','dead_flag_any',
            'exp_ratio','turn_ratio','mgmt_fee','actual_12b1',
            'first_offer_dt','fund_age_years','fsq_caldt','caldt_gap_days']
out_cols = [c for c in out_cols if c in df2.columns]
df2[out_cols].to_csv(PROC + 'final_step2_clean_wficn_edc_flows.csv', index=False)
print('\n  Saved → final_step2_clean_wficn_edc_flows.csv')

  Final EDC sample: 108,310 obs | 1,322 WFICNs | 1999-03-31 – 2011-12-30
  Flow winsorization bounds: [-0.1779, 0.2897]
  Obs with valid flow: 106,738 (98.5%)



  Saved → final_step2_clean_wficn_edc_flows.csv


## 5. QC Cleaning and Past Performance Construction

### 5.1 Final cleaning rules

Applied after the Step 2 sample construction:

- Drop observations where |portfolio_return| ≥ 1 (economically impossible for
  a diversified long-only equity fund; one CRSP recording error: mret = 12,012)
- Require lagged portfolio TNA ≥ \$10M (standard in the literature; removes
  extreme flow artefacts from micro funds changing size)
- Require non-null `flow_winsorized` (the dependent variable)

### 5.2 Past performance measures

Computed in log-space for numerical stability, with **gap-awareness**:

$$\text{past\_12m\_return}_t = \exp\!\left(\sum_{k=1}^{12} \ln(1+r_{t-k})\right) - 1$$

Key design choices:
- **shift(1) before rolling**: current month excluded from past return
- **Gap detection via run_id**: rolling window resets if there is a calendar gap
  in the fund's return series (prevents using non-consecutive months)
- **min_periods = window**: requires all 12 (or 6) months; NaN if fewer available

In [10]:
df3 = pd.read_csv(PROC + 'final_step2_clean_wficn_edc_flows.csv', low_memory=False)
df3['caldt']          = pd.to_datetime(df3['caldt'])
df3['first_offer_dt'] = pd.to_datetime(df3['first_offer_dt'], errors='coerce')

# ── Final QC cleaning ─────────────────────────────────────────────────────────
n0 = len(df3)

df3 = df3[df3['portfolio_return'].abs() < 1].copy()     # 1 CRSP error row
df3 = df3[df3['lag_portfolio_tna'].fillna(0) >= 10]     # ≥ $10M
df3 = df3[df3['flow_winsorized'].notna()].copy()         # need DV

print(f'  After QC: {len(df3):,} obs ({n0 - len(df3):,} dropped) | {df3["wficn"].nunique():,} WFICNs')

# ── Winsorize portfolio_return ────────────────────────────────────────────────
p01r = df3['portfolio_return'].quantile(0.01)
p99r = df3['portfolio_return'].quantile(0.99)
df3['portfolio_return_winsorized'] = df3['portfolio_return'].clip(p01r, p99r)

# ── Log-space past performance with gap-aware rolling ─────────────────────────
df3 = df3.sort_values(['wficn','caldt']).reset_index(drop=True)

df3['month_idx']     = df3['caldt'].dt.year * 12 + df3['caldt'].dt.month
df3['lag_month_idx'] = df3.groupby('wficn')['month_idx'].shift(1)
df3['month_gap']     = (df3['month_idx'] - df3['lag_month_idx']).fillna(1)
df3['is_break']      = (df3['month_gap'] != 1).astype(int)
# run_id increments at each gap; rolling within (wficn, run_id) resets at gaps
df3['run_id']        = df3.groupby('wficn')['is_break'].cumsum()
df3['log1p_ret']     = np.log1p(df3['portfolio_return_winsorized'])

def rolling_cumret(series, window):
    return series.shift(1).rolling(window, min_periods=window).sum()

grp = df3.groupby(['wficn','run_id'])
df3['sum_log_12m'] = grp['log1p_ret'].transform(lambda x: rolling_cumret(x, 12))
df3['sum_log_6m']  = grp['log1p_ret'].transform(lambda x: rolling_cumret(x,  6))
df3['past_12m_return'] = np.expm1(df3['sum_log_12m'])
df3['past_6m_return']  = np.expm1(df3['sum_log_6m'])

df3.drop(columns=['log1p_ret','sum_log_12m','sum_log_6m',
                  'is_break','run_id','lag_month_idx','month_idx','month_gap'],
         inplace=True)

# ── Control variables ─────────────────────────────────────────────────────────
df3['log_lag_tna']  = np.log(df3['lag_portfolio_tna'])
df3['log_fund_age'] = np.log1p(df3['fund_age_years'].clip(lower=0))

print(f'\n  past_12m_return: {df3["past_12m_return"].notna().sum():,} obs'
      f' | first available {df3.loc[df3["past_12m_return"].notna(),"caldt"].min().date()}')
print(f'  past_6m_return : {df3["past_6m_return"].notna().sum():,} obs'
      f' | first available {df3.loc[df3["past_6m_return"].notna(),"caldt"].min().date()}')

# Save Step 3 dataset
final_cols = ['wficn','caldt','flow_winsorized','flow','portfolio_return',
              'portfolio_return_winsorized','past_12m_return','past_6m_return',
              'portfolio_tna','lag_portfolio_tna','log_lag_tna',
              'fund_age_years','log_fund_age','exp_ratio','turn_ratio','mgmt_fee',
              'fund_name','crsp_obj_cd','lipper_obj_cd','lipper_obj_name',
              'n_share_classes']
final_cols = [c for c in final_cols if c in df3.columns]
df3[final_cols].to_csv(PROC + 'final_step3_regression_ready_wficn_edc.csv', index=False)
print('\n  Saved → final_step3_regression_ready_wficn_edc.csv')

  After QC: 99,325 obs (8,985 dropped) | 1,215 WFICNs



  past_12m_return: 83,186 obs | first available 2000-04-28
  past_6m_return : 90,877 obs | first available 1999-10-29



  Saved → final_step3_regression_ready_wficn_edc.csv


### Summary statistics — regression-ready dataset

In [11]:
df3_reg = pd.read_csv(PROC + 'final_step3_regression_ready_wficn_edc.csv', low_memory=False)
df3_reg['caldt'] = pd.to_datetime(df3_reg['caldt'])

stat_vars = {
    'flow_winsorized'            : 'Fund flow (winsorized)',
    'portfolio_return'           : 'Monthly portfolio return',
    'past_12m_return'            : 'Past 12-month return',
    'past_6m_return'             : 'Past 6-month return',
    'portfolio_tna'              : 'Portfolio TNA ($M)',
    'log_lag_tna'                : 'Log(lag TNA)',
    'exp_ratio'                  : 'Expense ratio',
    'turn_ratio'                 : 'Turnover ratio',
    'fund_age_years'             : 'Fund age (years)',
}
rows = []
for var, lbl in stat_vars.items():
    if var not in df3_reg.columns: continue
    s = df3_reg[var].dropna()
    rows.append({'Variable': lbl, 'N': f'{len(s):,}',
                 'Mean': f'{s.mean():.4f}', 'Std': f'{s.std():.4f}',
                 'p25': f'{s.quantile(0.25):.4f}', 'Median': f'{s.median():.4f}',
                 'p75': f'{s.quantile(0.75):.4f}'})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

                Variable      N     Mean       Std     p25   Median      p75
  Fund flow (winsorized) 99,325   0.0034    0.0523 -0.0152  -0.0028   0.0134
Monthly portfolio return 99,325   0.0067    0.0652 -0.0301   0.0115   0.0455
    Past 12-month return 83,186   0.0879    0.2451 -0.0693   0.1126   0.2416
     Past 6-month return 90,877   0.0437    0.1723 -0.0547   0.0533   0.1409
      Portfolio TNA ($M) 99,325 705.0744 1833.0153 73.7000 213.4000 611.7000
            Log(lag TNA) 99,325   5.3775    1.5046  4.2918   5.3571   6.4098
           Expense ratio 90,309   0.0134    0.0040  0.0110   0.0128   0.0153
          Turnover ratio 90,037   1.0024    1.0186  0.4600   0.7900   1.2700
        Fund age (years) 99,325  10.7980    8.4611  5.1636   8.9199  13.9138


## 6. Main 12-Month Flow-Performance Regressions

**Dependent variable:** `flow_winsorized`  
**Main explanatory variable:** `past_12m_return` (lagged; shift + rolling in log-space)

| Model | Specification |
|---|---|
| (1) Baseline | flow ~ past_12m_return |
| (2) Controls | Model 1 + log(TNA) + exp_ratio + fund_age_years + turn_ratio |
| (3) Month FE | Model 2 + month fixed effects (caldt period, drop first) |
| (4) Month+EDC FE | Model 3 + crsp_obj_cd fixed effects (EDCS = baseline) |

**Standard errors:** clustered by WFICN throughout (`cov_type='cluster'`).

**Preferred specification:** Model 3 (Month FE). Month fixed effects absorb all
time-series variation in aggregate flows, isolating the cross-sectional relationship.
The increase in the coefficient from Model 2 to Model 3 is expected: aggregate
market conditions drive both past returns and aggregate flows in parallel,
diluting the pooled estimate.

In [12]:
controls = ['log_lag_tna', 'exp_ratio', 'fund_age_years', 'turn_ratio']
required = ['flow_winsorized', 'past_12m_return'] + controls

reg12 = df3_reg.dropna(subset=required).copy().reset_index(drop=True)
reg12['month_str'] = reg12['caldt'].dt.to_period('M').astype(str)
m_dummies = pd.get_dummies(reg12['month_str'], prefix='m', drop_first=True, dtype=float)
e_dummies = pd.get_dummies(reg12['crsp_obj_cd'], prefix='edc', drop_first=False, dtype=float)
edc_cols  = [c for c in e_dummies.columns if c != 'edc_EDCS']
e_dummies = e_dummies[edc_cols]
m_dummies = m_dummies.reset_index(drop=True)
e_dummies = e_dummies.reset_index(drop=True)

Y12   = reg12['flow_winsorized']
cids  = reg12['wficn'].values
X12_1 = sm.add_constant(reg12['past_12m_return'])
X12_2 = sm.add_constant(reg12[['past_12m_return'] + controls])
X12_3 = sm.add_constant(pd.concat([reg12[['past_12m_return'] + controls], m_dummies], axis=1))
X12_4 = sm.add_constant(pd.concat([reg12[['past_12m_return'] + controls], m_dummies, e_dummies], axis=1))

print('Running 12m regressions …')
m12_1 = run_ols(Y12, X12_1, cids, '(1) Baseline         ')
m12_2 = run_ols(Y12, X12_2, cids, '(2) Controls         ')
m12_3 = run_ols(Y12, X12_3, cids, '(3) Month FE         ')
m12_4 = run_ols(Y12, X12_4, cids, '(4) Month + EDC FE   ')
models_12 = [m12_1, m12_2, m12_3, m12_4]
labels_4  = ['(1) Baseline', '(2) Controls', '(3) Month FE', '(4) Month+EDC FE']

Running 12m regressions …
  (1) Baseline           N=77,140  R²=0.0117  adj-R²=0.0117
  (2) Controls           N=77,140  R²=0.0229  adj-R²=0.0228


  (3) Month FE           N=77,140  R²=0.0867  adj-R²=0.0850


  (4) Month + EDC FE     N=77,140  R²=0.0891  adj-R²=0.0874


In [13]:
# ── Regression table ─────────────────────────────────────────────────────────
disp_vars = ['past_12m_return'] + controls + ['const']
disp_lbl  = {'past_12m_return': 'Past 12m Return', 'log_lag_tna': 'log(TNA_{t-1})',
             'exp_ratio': 'Expense Ratio', 'fund_age_years': 'Fund Age (years)',
             'turn_ratio': 'Turnover Ratio', 'const': 'Constant'}

rows = []
for var in disp_vars:
    lbl = disp_lbl.get(var, var)
    crow = {'Variable': lbl}; srow = {'Variable': ''}
    for i, m in enumerate(models_12, 1):
        if var in m.params:
            crow[f'M{i}'] = f'{m.params[var]:.4f}{stars(m.pvalues[var])}'
            srow[f'M{i}'] = f'({m.bse[var]:.4f})'
        else:
            crow[f'M{i}'] = '—'; srow[f'M{i}'] = ''
    rows += [crow, srow]
for lbl, vals in [('N',       [f'{int(m.nobs):,}' for m in models_12]),
                   ('R²',      [f'{m.rsquared:.4f}' for m in models_12]),
                   ('Month FE',['No','No','Yes','Yes']),
                   ('EDC FE',  ['No','No','No','Yes'])]:
    r = {'Variable': lbl}
    for i,v in enumerate(vals,1): r[f'M{i}'] = v
    rows.append(r)
tbl12 = pd.DataFrame(rows, columns=['Variable','M1','M2','M3','M4'])
tbl12.columns = ['Variable'] + labels_4

print(tbl12.to_string(index=False))
print('\nNotes: DV=flow_winsorized. WFICN-clustered SEs. *** p<0.01 ** p<0.05 * p<0.10')

tbl12.to_csv(TABS + 'final_step4_main_regressions_12m.csv', index=False)
print('\nSaved → final_step4_main_regressions_12m.csv')

        Variable (1) Baseline (2) Controls (3) Month FE (4) Month+EDC FE
 Past 12m Return    0.0209***    0.0201***    0.0928***        0.0943***
                     (0.0013)     (0.0013)     (0.0047)         (0.0046)
  log(TNA_{t-1})            —       0.0002       0.0003           0.0002
                                  (0.0003)     (0.0003)         (0.0003)
   Expense Ratio            —    -0.2551**    -0.2477**          -0.1499
                                  (0.1053)     (0.1109)         (0.1095)
Fund Age (years)            —   -0.0006***   -0.0004***       -0.0004***
                                  (0.0001)     (0.0001)         (0.0001)
  Turnover Ratio            —   -0.0016***    -0.0013**       -0.0015***
                                  (0.0005)     (0.0005)         (0.0005)
        Constant      -0.0002    0.0101***   -0.0342***       -0.0360***
                     (0.0004)     (0.0025)     (0.0041)         (0.0040)
               N       77,140       77,140       77

### Economic interpretation — preferred model (3)

In [14]:
c3   = m12_3.params['past_12m_return']
se3  = m12_3.bse['past_12m_return']
t3   = m12_3.tvalues['past_12m_return']
p3   = m12_3.pvalues['past_12m_return']
sd_p = reg12['past_12m_return'].std()
sd_f = Y12.std()

print(f'Model 3 — past_12m_return coefficient: {c3:+.4f}{stars(p3)}')
print(f'  SE={se3:.4f}  t={t3:.2f}  p={p3:.4f}')
print(f'\n+10 pp past 12m return → {c3*0.10*100:+.3f} pp/month more flow')
print(f'+1 SD past return ({sd_p*100:.1f} pp) → {c3*sd_p*100:+.3f} pp flow ({abs(c3*sd_p)/sd_f:.2f} SDs of flow)')

Model 3 — past_12m_return coefficient: +0.0928***
  SE=0.0047  t=19.85  p=0.0000

+10 pp past 12m return → +0.928 pp/month more flow
+1 SD past return (24.4 pp) → +2.265 pp flow (0.48 SDs of flow)


## 7. 12-Month Quintile Flow-Performance Analysis

Each calendar month, funds are sorted into quintiles based on `past_12m_return`:
- **Q1** = bottom 20% (worst past performers)
- **Q5** = top 20% (best past performers)
- **Q1 is the reference (omitted) category** in all regressions

This specification directly tests for:
1. **Monotonicity**: do Q2–Q5 coefficients increase from bottom to top?
2. **Convexity**: is the Q4→Q5 step disproportionately large relative to Q1–Q4?

In [15]:
def assign_quintile(series):
    try:
        return pd.qcut(series, q=5, labels=[1,2,3,4,5]).astype(int)
    except ValueError:
        return pd.qcut(series.rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)

df_q12 = df3_reg.dropna(subset=['flow_winsorized','past_12m_return']).copy()
df_q12['quintile'] = df_q12.groupby('caldt')['past_12m_return'].transform(assign_quintile)

# ── Average flows by quintile ─────────────────────────────────────────────────
flow_q12 = (
    df_q12.groupby('quintile')['flow_winsorized']
    .agg(mean_flow='mean', median_flow='median', n_obs='count')
    .reset_index()
)
flow_q12['unique_wficn'] = df_q12.groupby('quintile')['wficn'].nunique().values
flow_q12['quintile_label'] = flow_q12['quintile'].map(
    {1:'Q1 (worst)',2:'Q2',3:'Q3',4:'Q4',5:'Q5 (best)'})

q5m = flow_q12.loc[flow_q12['quintile']==5,'mean_flow'].values[0]
q1m = flow_q12.loc[flow_q12['quintile']==1,'mean_flow'].values[0]
spread12 = q5m - q1m

print('Average monthly flows by 12m performance quintile:')
print(flow_q12[['quintile_label','mean_flow','median_flow','n_obs','unique_wficn']].to_string(index=False))
print(f'\nQ5 − Q1 spread: {spread12:+.4f} ({spread12*100:+.3f} pp/month)')

flow_q12.to_csv(TABS + 'final_step5_quintile_average_flows.csv', index=False)

Average monthly flows by 12m performance quintile:
quintile_label  mean_flow  median_flow  n_obs  unique_wficn
    Q1 (worst)  -0.013521    -0.011621  16694           964
            Q2  -0.005698    -0.006688  16606          1045
            Q3  -0.000859    -0.003521  16613          1045
            Q4   0.005690    -0.000216  16606          1025
     Q5 (best)   0.018667     0.005797  16667           909

Q5 − Q1 spread: +0.0322 (+3.219 pp/month)


In [16]:
# ── Quintile regressions ─────────────────────────────────────────────────────
regq12 = df_q12.dropna(subset=controls).copy().reset_index(drop=True)

qd = pd.get_dummies(regq12['quintile'].astype(int), prefix='Q', drop_first=False, dtype=float)
qd = qd.drop(columns=['Q_1']).reset_index(drop=True)  # Q1 = reference
q_cols = ['Q_2','Q_3','Q_4','Q_5']

regq12['month_str'] = regq12['caldt'].dt.to_period('M').astype(str)
mq = pd.get_dummies(regq12['month_str'], prefix='m', drop_first=True, dtype=float).reset_index(drop=True)
eq = pd.get_dummies(regq12['crsp_obj_cd'], prefix='edc', drop_first=False, dtype=float).reset_index(drop=True)
eq = eq[[c for c in eq.columns if c != 'edc_EDCS']]

Yq   = regq12['flow_winsorized']
cidsq = regq12['wficn'].values
Xq1  = sm.add_constant(qd)
Xq2  = sm.add_constant(pd.concat([qd, regq12[controls]], axis=1))
Xq3  = sm.add_constant(pd.concat([qd, regq12[controls], mq], axis=1))
Xq4  = sm.add_constant(pd.concat([qd, regq12[controls], mq, eq], axis=1))

print('Running 12m quintile regressions …')
mq1 = run_ols(Yq, Xq1, cidsq, '(1) Quintile dummies      ')
mq2 = run_ols(Yq, Xq2, cidsq, '(2) + Controls            ')
mq3 = run_ols(Yq, Xq3, cidsq, '(3) + Month FE            ')
mq4 = run_ols(Yq, Xq4, cidsq, '(4) + Month FE + EDC FE   ')
models_q12 = [mq1,mq2,mq3,mq4]

# Table
rowsq = []
for var in q_cols + controls + ['const']:
    lbl = disp_lbl.get(var, var.replace('Q_','Q'))
    cr = {'Variable': lbl}; sr = {'Variable': ''}
    for i, m in enumerate(models_q12, 1):
        if var in m.params:
            cr[f'M{i}'] = f'{m.params[var]:.4f}{stars(m.pvalues[var])}'
            sr[f'M{i}'] = f'({m.bse[var]:.4f})'
        else:
            cr[f'M{i}'] = '—'; sr[f'M{i}'] = ''
    rowsq += [cr, sr]
for lbl, vals in [('Q1 (ref.)',['Yes']*4),
                   ('N',       [f'{int(m.nobs):,}' for m in models_q12]),
                   ('R²',      [f'{m.rsquared:.4f}' for m in models_q12]),
                   ('Month FE',['No','No','Yes','Yes']),
                   ('EDC FE',  ['No','No','No','Yes'])]:
    r = {'Variable': lbl}
    for i,v in enumerate(vals,1): r[f'M{i}'] = v
    rowsq.append(r)
tblq12 = pd.DataFrame(rowsq, columns=['Variable','M1','M2','M3','M4'])
tblq12.columns = ['Variable'] + labels_4
print(tblq12.to_string(index=False))
tblq12.to_csv(TABS + 'final_step5_quintile_regressions.csv', index=False)

Running 12m quintile regressions …
  (1) Quintile dummies        N=77,140  R²=0.0565  adj-R²=0.0565
  (2) + Controls              N=77,140  R²=0.0657  adj-R²=0.0656


  (3) + Month FE              N=77,140  R²=0.0936  adj-R²=0.0919


  (4) + Month FE + EDC FE     N=77,140  R²=0.0957  adj-R²=0.0939
        Variable (1) Baseline (2) Controls (3) Month FE (4) Month+EDC FE
              Q2    0.0082***    0.0078***    0.0077***        0.0076***
                     (0.0007)     (0.0007)     (0.0007)         (0.0007)
              Q3    0.0131***    0.0124***    0.0122***        0.0123***
                     (0.0007)     (0.0007)     (0.0007)         (0.0007)
              Q4    0.0198***    0.0191***    0.0189***        0.0190***
                     (0.0008)     (0.0009)     (0.0008)         (0.0008)
              Q5    0.0331***    0.0324***    0.0323***        0.0325***
                     (0.0012)     (0.0012)     (0.0012)         (0.0012)
  log(TNA_{t-1})            —       0.0002       0.0001           0.0000
                                  (0.0003)     (0.0003)         (0.0003)
   Expense Ratio            —      -0.1491    -0.2337**          -0.1488
                                  (0.0995)     (0.1087)    

In [17]:
# ── Monotonicity and convexity check ─────────────────────────────────────────
coefs3q = [mq3.params[q] for q in q_cols]
diffs3q  = [coefs3q[i+1] - coefs3q[i] for i in range(len(coefs3q)-1)]
mono3q   = all(d > 0 for d in diffs3q)
coefs_full = [0.0] + coefs3q
q4q5_step  = coefs_full[4] - coefs_full[3]
avg_inner  = np.mean([coefs_full[2]-coefs_full[1], coefs_full[3]-coefs_full[2]])

print(f'Model 3 quintile coefficients vs Q1:')
for q, c in zip(q_cols, coefs3q):
    p = mq3.pvalues[q]
    print(f'  {q.replace("_","")}: {c:+.4f}{stars(p)}  ({c*100:+.3f} pp/month)')
print(f'\nMonotonically increasing Q2→Q5: {"YES" if mono3q else "NO"}')
print(f'Q4→Q5 step: {q4q5_step:+.4f} vs avg inner step: {avg_inner:+.4f} → convexity ratio: {q4q5_step/avg_inner:.2f}x')
print(f'Interpretation: {"Convex (disproportionate top-quintile flow)" if q4q5_step > avg_inner else "Linear"}')

Model 3 quintile coefficients vs Q1:
  Q2: +0.0077***  (+0.765 pp/month)
  Q3: +0.0122***  (+1.225 pp/month)
  Q4: +0.0189***  (+1.893 pp/month)
  Q5: +0.0323***  (+3.225 pp/month)

Monotonically increasing Q2→Q5: YES
Q4→Q5 step: +0.0133 vs avg inner step: +0.0056 → convexity ratio: 2.36x
Interpretation: Convex (disproportionate top-quintile flow)


## 8. 6-Month Robustness Check

Replaces `past_12m_return` with `past_6m_return` to verify that the
flow-performance finding is not specific to the 12-month horizon.

The 6-month sample has more observations (the shorter window requires fewer
consecutive months before first becoming available).

**Part A:** Continuous performance regressions (same 4 models)
**Part B:** Monthly quintile sort on `past_6m_return`

In [18]:
# ── Part A: continuous 6m regressions ────────────────────────────────────────
req6 = ['flow_winsorized','past_6m_return'] + controls
reg6 = df3_reg.dropna(subset=req6).copy().reset_index(drop=True)
reg6['month_str'] = reg6['caldt'].dt.to_period('M').astype(str)
m6d = pd.get_dummies(reg6['month_str'], prefix='m', drop_first=True, dtype=float).reset_index(drop=True)
e6d = pd.get_dummies(reg6['crsp_obj_cd'], prefix='edc', drop_first=False, dtype=float).reset_index(drop=True)
e6d = e6d[[c for c in e6d.columns if c != 'edc_EDCS']]

Y6   = reg6['flow_winsorized']
cids6 = reg6['wficn'].values
X6_1 = sm.add_constant(reg6['past_6m_return'])
X6_2 = sm.add_constant(reg6[['past_6m_return'] + controls])
X6_3 = sm.add_constant(pd.concat([reg6[['past_6m_return'] + controls], m6d], axis=1))
X6_4 = sm.add_constant(pd.concat([reg6[['past_6m_return'] + controls], m6d, e6d], axis=1))

print('Running 6m robustness regressions …')
m6_1 = run_ols(Y6, X6_1, cids6, '(1) Baseline ')
m6_2 = run_ols(Y6, X6_2, cids6, '(2) Controls ')
m6_3 = run_ols(Y6, X6_3, cids6, '(3) Month FE ')
m6_4 = run_ols(Y6, X6_4, cids6, '(4) Month+EDC')
models_6 = [m6_1,m6_2,m6_3,m6_4]

rows6 = []
for var in ['past_6m_return'] + controls + ['const']:
    lbl = disp_lbl.get(var, var)
    if lbl == var: lbl = 'Past 6m Return' if var == 'past_6m_return' else var
    cr = {'Variable': lbl}; sr = {'Variable': ''}
    for i, m in enumerate(models_6, 1):
        if var in m.params:
            cr[f'M{i}'] = f'{m.params[var]:.4f}{stars(m.pvalues[var])}'
            sr[f'M{i}'] = f'({m.bse[var]:.4f})'
        else:
            cr[f'M{i}'] = '—'; sr[f'M{i}'] = ''
    rows6 += [cr, sr]
for lbl, vals in [('N',       [f'{int(m.nobs):,}' for m in models_6]),
                   ('R²',      [f'{m.rsquared:.4f}' for m in models_6]),
                   ('Month FE',['No','No','Yes','Yes']),
                   ('EDC FE',  ['No','No','No','Yes'])]:
    r = {'Variable': lbl}
    for i,v in enumerate(vals,1): r[f'M{i}'] = v
    rows6.append(r)
tbl6 = pd.DataFrame(rows6, columns=['Variable','M1','M2','M3','M4'])
tbl6.columns = ['Variable'] + labels_4
print(tbl6.to_string(index=False))
tbl6.to_csv(TABS + 'final_step6_robustness_6m_regressions.csv', index=False)

Running 6m robustness regressions …
  (1) Baseline   N=83,319  R²=0.0157  adj-R²=0.0157
  (2) Controls   N=83,319  R²=0.0288  adj-R²=0.0287


  (3) Month FE   N=83,319  R²=0.0805  adj-R²=0.0789


  (4) Month+EDC  N=83,319  R²=0.0827  adj-R²=0.0810
        Variable (1) Baseline (2) Controls (3) Month FE (4) Month+EDC FE
  Past 6m Return    0.0362***    0.0358***    0.1319***        0.1334***
                     (0.0018)     (0.0018)     (0.0058)         (0.0058)
  log(TNA_{t-1})            —       0.0001       0.0002           0.0001
                                  (0.0003)     (0.0003)         (0.0003)
   Expense Ratio            —     -0.1856*     -0.2069*          -0.1176
                                  (0.1052)     (0.1068)         (0.1079)
Fund Age (years)            —   -0.0007***   -0.0006***       -0.0006***
                                  (0.0001)     (0.0001)         (0.0001)
  Turnover Ratio            —      -0.0008     -0.0010*         -0.0012*
                                  (0.0006)     (0.0006)         (0.0006)
        Constant    0.0013***    0.0112***      -0.0052        -0.0067**
                     (0.0004)     (0.0025)     (0.0033)         (0.0033)

In [19]:
# ── Part B: 6m quintile analysis ─────────────────────────────────────────────
df_q6 = df3_reg.dropna(subset=['flow_winsorized','past_6m_return']).copy()
df_q6['quintile_6m'] = df_q6.groupby('caldt')['past_6m_return'].transform(assign_quintile)

flow_q6 = (
    df_q6.groupby('quintile_6m')['flow_winsorized']
    .agg(mean_flow='mean', median_flow='median', n_obs='count')
    .reset_index()
)
flow_q6['unique_wficn'] = df_q6.groupby('quintile_6m')['wficn'].nunique().values
flow_q6['label'] = flow_q6['quintile_6m'].map({1:'Q1 (worst)',2:'Q2',3:'Q3',4:'Q4',5:'Q5 (best)'})
q5_6m = flow_q6.loc[flow_q6['quintile_6m']==5,'mean_flow'].values[0]
q1_6m = flow_q6.loc[flow_q6['quintile_6m']==1,'mean_flow'].values[0]
spread6 = q5_6m - q1_6m

print('Average monthly flows by 6m performance quintile:')
print(flow_q6[['label','mean_flow','median_flow','n_obs']].to_string(index=False))
print(f'\nQ5 − Q1 spread (6m): {spread6:+.4f} ({spread6*100:+.3f} pp/month)')
flow_q6.to_csv(TABS + 'final_step6_robustness_6m_quintile_average_flows.csv', index=False)

# 6m quintile regressions
regq6 = df_q6.dropna(subset=controls).copy().reset_index(drop=True)
q6d = pd.get_dummies(regq6['quintile_6m'].astype(int), prefix='Q', drop_first=False, dtype=float)
q6d = q6d.drop(columns=['Q_1']).reset_index(drop=True)
regq6['month_str'] = regq6['caldt'].dt.to_period('M').astype(str)
mq6d = pd.get_dummies(regq6['month_str'], prefix='m', drop_first=True, dtype=float).reset_index(drop=True)
eq6d = pd.get_dummies(regq6['crsp_obj_cd'], prefix='edc', drop_first=False, dtype=float).reset_index(drop=True)
eq6d = eq6d[[c for c in eq6d.columns if c != 'edc_EDCS']]
Yq6   = regq6['flow_winsorized']
cids_q6 = regq6['wficn'].values

print('\nRunning 6m quintile regressions …')
mq6_1 = run_ols(Yq6, sm.add_constant(q6d), cids_q6, '(1)')
mq6_2 = run_ols(Yq6, sm.add_constant(pd.concat([q6d, regq6[controls]], axis=1)), cids_q6, '(2)')
mq6_3 = run_ols(Yq6, sm.add_constant(pd.concat([q6d, regq6[controls], mq6d], axis=1)), cids_q6, '(3)')
mq6_4 = run_ols(Yq6, sm.add_constant(pd.concat([q6d, regq6[controls], mq6d, eq6d], axis=1)), cids_q6, '(4)')
models_q6 = [mq6_1,mq6_2,mq6_3,mq6_4]

rowsq6 = []
for var in q_cols + controls + ['const']:
    lbl = disp_lbl.get(var, var.replace('Q_','Q'))
    cr = {'Variable': lbl}; sr = {'Variable': ''}
    for i, m in enumerate(models_q6, 1):
        if var in m.params:
            cr[f'M{i}'] = f'{m.params[var]:.4f}{stars(m.pvalues[var])}'
            sr[f'M{i}'] = f'({m.bse[var]:.4f})'
        else:
            cr[f'M{i}'] = '—'; sr[f'M{i}'] = ''
    rowsq6 += [cr, sr]
for lbl, vals in [('Q1 (ref.)',['Yes']*4),
                   ('N',       [f'{int(m.nobs):,}' for m in models_q6]),
                   ('R²',      [f'{m.rsquared:.4f}' for m in models_q6]),
                   ('Month FE',['No','No','Yes','Yes']),
                   ('EDC FE',  ['No','No','No','Yes'])]:
    r = {'Variable': lbl}
    for i,v in enumerate(vals,1): r[f'M{i}'] = v
    rowsq6.append(r)
tblq6 = pd.DataFrame(rowsq6, columns=['Variable','M1','M2','M3','M4'])
tblq6.columns = ['Variable'] + labels_4
print(tblq6.to_string(index=False))
tblq6.to_csv(TABS + 'final_step6_robustness_6m_quintile_regressions.csv', index=False)

Average monthly flows by 6m performance quintile:
     label  mean_flow  median_flow  n_obs
Q1 (worst)  -0.010135    -0.009638  18231
        Q2  -0.003710    -0.005724  18149
        Q3   0.000591    -0.003165  18143
        Q4   0.005134    -0.000801  18149
 Q5 (best)   0.017789     0.004127  18205

Q5 − Q1 spread (6m): +0.0279 (+2.792 pp/month)

Running 6m quintile regressions …
  (1)  N=83,319  R²=0.0395  adj-R²=0.0394
  (2)  N=83,319  R²=0.0518  adj-R²=0.0518


  (3)  N=83,319  R²=0.0784  adj-R²=0.0767


  (4)  N=83,319  R²=0.0802  adj-R²=0.0785
        Variable (1) Baseline (2) Controls (3) Month FE (4) Month+EDC FE
              Q2    0.0067***    0.0066***    0.0064***        0.0064***
                     (0.0006)     (0.0006)     (0.0006)         (0.0006)
              Q3    0.0112***    0.0108***    0.0107***        0.0107***
                     (0.0007)     (0.0007)     (0.0007)         (0.0007)
              Q4    0.0159***    0.0156***    0.0155***        0.0155***
                     (0.0008)     (0.0008)     (0.0008)         (0.0007)
              Q5    0.0291***    0.0287***    0.0286***        0.0288***
                     (0.0012)     (0.0011)     (0.0011)         (0.0011)
  log(TNA_{t-1})            —       0.0004       0.0003           0.0002
                                  (0.0003)     (0.0003)         (0.0003)
   Expense Ratio            —      -0.0917     -0.1838*          -0.1038
                                  (0.1009)     (0.1066)         (0.1084)
Fund Age 

## 9. Final Overview and QC

Summary of sample sizes, key results, and consistency checks.

In [20]:
print('=' * 66)
print('SAMPLE SIZE FUNNEL')
print('=' * 66)

funnel = [
    ('Step 1 — All WFICN-months',       agg),
    ('Step 2 — Active EDC filtered',    df2),
    ('Step 3 — Regression-ready',       df3_reg),
]
# Derived samples
req12_ = ['flow_winsorized','past_12m_return'] + controls
req6_  = ['flow_winsorized','past_6m_return']  + controls
r12_s  = df3_reg.dropna(subset=req12_)
r6_s   = df3_reg.dropna(subset=req6_)

for label, d in funnel:
    print(f'  {label:<38}: {len(d):>10,} obs | {d["wficn"].nunique():>6,} WFICNs'
          f' | {d["caldt"].min().date()} – {d["caldt"].max().date()}')
print(f'  {"12m regression sample (all controls)":<38}: {len(r12_s):>10,} obs | {r12_s["wficn"].nunique():>6,} WFICNs')
print(f'  {"6m  regression sample (all controls)":<38}: {len(r6_s):>10,} obs | {r6_s["wficn"].nunique():>6,} WFICNs')

print()
print('=' * 66)
print('KEY RESULTS SUMMARY')
print('=' * 66)

c12_3 = m12_3.params['past_12m_return']
c6_3  = m6_3.params['past_6m_return']

print(f'\n12m continuous (Model 3):  coef={c12_3:+.4f}{stars(m12_3.pvalues["past_12m_return"])}  '
      f'+10pp → {c12_3*0.10*100:+.3f} pp/month  R²={m12_3.rsquared:.4f}')
print(f' 6m continuous (Model 3):  coef={c6_3:+.4f}{stars(m6_3.pvalues["past_6m_return"])}  '
      f'+10pp → {c6_3*0.10*100:+.3f} pp/month  R²={m6_3.rsquared:.4f}')
print(f'\n12m Q5 premium (Model 3): {mq3.params["Q_5"]*100:+.3f} pp/month  monotone={'YES' if mono3q else 'NO'}  convexity={q4q5_step/avg_inner:.2f}×')

q5_adj_6m = mq6_3.params['Q_5']
c6q = [mq6_3.params[q] for q in q_cols]
mono6 = all(c6q[i+1]>c6q[i] for i in range(len(c6q)-1))
cf6  = [0.0]+c6q
r6_cv = (cf6[4]-cf6[3])/np.mean([cf6[2]-cf6[1],cf6[3]-cf6[2]])
print(f' 6m Q5 premium (Model 3): {q5_adj_6m*100:+.3f} pp/month  monotone={"YES" if mono6 else "NO"}  convexity={r6_cv:.2f}×')
print()
print('All specifications positive, significant (p<0.001), monotone, and convex.')

SAMPLE SIZE FUNNEL
  Step 1 — All WFICN-months             :  1,241,561 obs | 10,740 WFICNs | 1991-01-31 – 2011-12-30
  Step 2 — Active EDC filtered          :    108,310 obs |  1,322 WFICNs | 1999-03-31 – 2011-12-30
  Step 3 — Regression-ready             :     99,325 obs |  1,215 WFICNs | 1999-04-30 – 2011-12-30
  12m regression sample (all controls)  :     77,140 obs |    909 WFICNs
  6m  regression sample (all controls)  :     83,319 obs |    957 WFICNs

KEY RESULTS SUMMARY

12m continuous (Model 3):  coef=+0.0928***  +10pp → +0.928 pp/month  R²=0.0867
 6m continuous (Model 3):  coef=+0.1319***  +10pp → +1.319 pp/month  R²=0.0805

12m Q5 premium (Model 3): +3.225 pp/month  monotone=YES  convexity=2.36×
 6m Q5 premium (Model 3): +2.861 pp/month  monotone=YES  convexity=2.91×

All specifications positive, significant (p<0.001), monotone, and convex.


## 10. Extension: 2003–2025 Recent-Sample Analysis

### Overview

This section applies the **same empirical pipeline** used in the baseline analysis to a
more recent CRSP sample covering 2003–2025. It constitutes the main **own contribution**
of this thesis.

**Methodology (identical to baseline):**
- WFICN-level TNA-weighted return aggregation (MFLinks mapping)
- EDC filter (domestic active equity), ETF/index fund exclusions
- Flow = TNA_t / TNA_{t-1} − (1 + R_t), gap-validated, winsorised at 1st/99th pct
- Past 12-month return: rolling gap-aware cumulative return, lagged one period
- OLS with WFICN-clustered standard errors; Month FE (M3) and Month + EDC FE (M4)

**Why 2003–2025?**  
The 2003 starting year reflects the availability of the new CRSP data extract.
The extension provides temporal robustness and tests whether the performance-chasing
pattern documented in the baseline (1991–2011) persists in a materially different
market environment including the Global Financial Crisis (2008–2009), the COVID-19
shock (2020), and the 2022 interest rate cycle.

**Burn-in note:**  
Although raw CRSP data begin in 2003, the main 12-month regression sample begins
in **2004** because `past_12m_return` requires 12 consecutive lagged monthly returns.

**Subperiod analysis** is not conducted due to the 20-page thesis limit.

**Extension scripts** (run separately, outputs loaded below):
- `extension_2003_2025_step1_wficn_portfolio_month.py`
- `extension_2003_2025_step2_clean_edc_flows.py`
- `extension_2003_2025_step3_regressions_quintiles.py`
- `extension_2003_2025_final_qc_comparison.py`


### 10.1 Extension Sample Construction


In [21]:
import pandas as pd, os
# EXT_TABS points to the output/tables directory (works from notebooks/ or project root)
EXT_TABS = os.path.normpath(os.path.join(os.getcwd(), "..", "output", "tables"))

sc_ext = pd.read_csv(os.path.join(EXT_TABS, "extension_2003_2025_sample_construction.csv"))
print("Extension Sample Construction")
print("=" * 70)
funnel_cols = ["step", "obs", "unique_wficn", "note"]
disp_cols = [c for c in funnel_cols if c in sc_ext.columns]
print(sc_ext[disp_cols].to_string(index=False))
print()
print("Key numbers:")
print("  Full cleaned sample:          193,074 obs, 1,474 unique WFICNs")
print("  12m regression sample:        135,585 obs (after requiring exp_ratio + turn_ratio)")
print("  Effective regression period:  2004-01 to 2025-12")


Extension Sample Construction
                                                          step       obs  unique_wficn                                          note
                   0. All WFICN-month observations after merge 2115074.0       14421.0                                           NaN
                            1. Drop missing/zero portfolio_tna 2114869.0       14421.0                positive TNA required for flow
                              2. Drop missing portfolio_return 2060322.0       14415.0                           no return available
             3. Keep EDC funds (crsp_obj_cd starts with 'EDC')  280439.0        2124.0  broader 'ED' prefix would keep 1,073,066 obs
                                 4. Exclude ETFs (flag + name)  253355.0        1881.0                  flag: 27,012  name-based: 72
                          5. Exclude index funds (flag + name)  210421.0        1612.0               flag: 41,253  name-based: 1,681
6. Exclude non-target types (internatio

### 10.2 Extension Summary Statistics


In [22]:
ss_ext = pd.read_csv(os.path.join(EXT_TABS, 'extension_2003_2025_summary_statistics.csv'))

# Select and format key variables
key_vars = [
    'Fund flow (winsorized 1/99)',
    'Monthly portfolio return',
    'Past 12-month cumulative return',
    'Portfolio TNA ($M)',
    'Expense ratio',
    'Turnover ratio',
    'Fund age (years)',
]
ss_disp = ss_ext[ss_ext['variable'].isin(key_vars)].copy()
ss_disp = ss_disp.set_index('variable')

# Convert flow/return to percentage points where appropriate
pct_vars = [
    'Fund flow (winsorized 1/99)',
    'Monthly portfolio return',
    'Past 12-month cumulative return',
    'Expense ratio',
]
for v in pct_vars:
    if v in ss_disp.index:
        for col in ['mean', 'std', 'p25', 'median', 'p75']:
            ss_disp.loc[v, col] = ss_disp.loc[v, col] * 100

print('Extension Summary Statistics (2003–2025, active EDC funds)')
print('  Flow/return variables in percentage points; TNA in $M')
print('=' * 85)
print(ss_disp[['N', 'mean', 'std', 'p25', 'median', 'p75']].round(2).to_string())


Extension Summary Statistics (2003–2025, active EDC funds)
  Flow/return variables in percentage points; TNA in $M
                                      N     mean      std    p25  median     p75
variable                                                                        
Fund flow (winsorized 1/99)      193074    -0.28     4.08  -1.49   -0.56    0.48
Monthly portfolio return         193074     0.91     5.48  -2.08    1.31    4.20
Past 12-month cumulative return  170616    11.07    19.11  -0.30   10.72   21.77
Portfolio TNA ($M)               193074  1078.51  2667.08  99.60  307.50  909.20
Expense ratio                    153413     1.29     0.38   1.07    1.26    1.50
Turnover ratio                   153174     0.75     0.67   0.34    0.60    0.96
Fund age (years)                 193074    16.18    10.20   8.59   14.78   21.92


### 10.3 Extension Main 12-Month Flow-Performance Regressions

Same four specifications as the baseline (Section 6).  
Dependent variable: `flow_winsorized`. Main regressor: `past_12m_return`.
Standard errors clustered by WFICN.


In [23]:
reg_ext = pd.read_csv(os.path.join(EXT_TABS, 'extension_2003_2025_main_regressions_12m.csv'))
print('Extension: 12-Month Flow-Performance Regressions')
print('=' * 70)
print(reg_ext.to_string(index=False))
print()
print('Preferred specification (M3, month FE):')
perf_row = reg_ext[reg_ext['Variable'] == 'Past 12m Return']
if len(perf_row) > 0:
    coef_m3 = perf_row['(3) Month FE'].values[0]
    print(f"  Coefficient on past_12m_return: {coef_m3}")
    print(f"  Interpretation: +10 pp higher past return → +1.171 pp/month higher flow")
print()
# Economic interpretation
coef = 0.1171
effect_10pp = coef * 0.10 * 100
print(f"  +10 pp past return → +{effect_10pp:.3f} pp/month flow change")
print(f"  Significant at p<0.001 in all four specifications")


Extension: 12-Month Flow-Performance Regressions
        Variable (1) Baseline (2) Controls (3) Month FE (4) Month+EDC FE
 Past 12m Return    0.0219***    0.0216***    0.1171***        0.1173***
             NaN     (0.0011)     (0.0011)     (0.0043)         (0.0043)
  log(TNA_{t-1})            —       0.0000      -0.0001          -0.0002
             NaN          NaN     (0.0002)     (0.0002)         (0.0002)
   Expense Ratio            —   -0.3297***   -0.3787***       -0.3293***
             NaN          NaN     (0.0754)     (0.0833)         (0.0827)
Fund Age (years)            —   -0.0004***   -0.0003***       -0.0003***
             NaN          NaN     (0.0000)     (0.0000)         (0.0000)
  Turnover Ratio            —   -0.0013***   -0.0014***       -0.0016***
             NaN          NaN     (0.0005)     (0.0005)         (0.0005)
        Constant   -0.0061***    0.0057***   -0.0405***       -0.0406***
             NaN     (0.0003)     (0.0018)     (0.0035)         (0.0035)
  

### 10.4 Extension Quintile Analysis

Monthly quintile sort on `past_12m_return`; Q1 = reference group.


In [24]:
# Panel A: average flows
qavg_ext = pd.read_csv(os.path.join(EXT_TABS, 'extension_2003_2025_quintile_average_flows.csv'))
print('Extension Quintile Analysis — Panel A: Average Monthly Flows')
print('=' * 70)
if 'Mean Flow (pp)' in qavg_ext.columns:
    disp = qavg_ext[['Quintile', 'Mean Flow (pp)', 'Median Flow (pp)', 'N obs']]
else:
    disp = qavg_ext
print(disp.to_string(index=False))

print()
print('Extension Quintile Analysis — Panel B: Regression Coefficients')
print('=' * 70)
qreg_ext = pd.read_csv(os.path.join(EXT_TABS, 'extension_2003_2025_quintile_regressions.csv'))
print(qreg_ext.to_string(index=False))
print()
print('Key metrics (M3):')
print('  Q5 premium (reg-adjusted):  +2.61 pp/month')
print('  Q5−Q1 unconditional spread: +2.23 pp/month')
print('  Convexity ratio:             3.10×')
print('  Monotonic Q1→Q5:            Yes (all 4 models)')


Extension Quintile Analysis — Panel A: Average Monthly Flows
      Quintile  Mean Flow (pp)  Median Flow (pp)   N obs
    Q1 (worst)         -1.4970           -1.1252 34231.0
            Q2         -0.8368           -0.7808 34075.0
            Q3         -0.5088           -0.6186 34064.0
            Q4         -0.1911           -0.4739 34075.0
     Q5 (best)          0.7375           -0.0950 34171.0
Q5 − Q1 spread          2.2345               NaN     NaN

Extension Quintile Analysis — Panel B: Regression Coefficients
        Variable (1) Baseline (2) Controls (3) Month FE (4) Month+EDC FE
              Q2    0.0081***    0.0076***    0.0075***        0.0075***
             NaN     (0.0005)     (0.0005)     (0.0005)         (0.0004)
              Q3    0.0117***    0.0112***    0.0111***        0.0110***
             NaN     (0.0005)     (0.0005)     (0.0005)         (0.0005)
              Q4    0.0153***    0.0148***    0.0148***        0.0147***
             NaN     (0.0006)     (0.0

### 10.5 Baseline vs. Extension Comparison

Both samples use identical methodology. Differences reflect the fund universe
and market environment changes across periods.


In [25]:
comp = pd.read_csv(os.path.join(EXT_TABS, 'extension_2003_2025_baseline_comparison.csv'))
print('Baseline vs. Extension Comparison')
print('=' * 75)
print(comp.to_string(index=False))
print()
print('Interpretation:')
print('  1. Positive flow-performance relationship confirmed in both samples (p<0.001)')
print('  2. Extension coefficient (+0.1171) is 26% stronger than baseline (+0.0928)')
print('     Plausible given heightened return dispersion in GFC/COVID/2022 rate cycle')
print('  3. Q5 premium slightly smaller in extension (+2.61 vs +3.23 pp/month)')
print('     Larger, more diverse fund universe in extension period')
print('  4. Convexity stronger in extension (3.10× vs ~2.4×)')
print('     Disproportionate reward for top performers persists and intensifies')
print('  5. All 14 consistency checks passed (see extension_2003_2025_final_qc_comparison.py)')


Baseline vs. Extension Comparison
                        Metric              Baseline (1991–2011)             Extension (2003–2025)
                  Sample label              Baseline replication      Extension (own contribution)
               Raw CRSP period                         1991–2011                         2003–2025
     Effective 12m reg. period                         1999–2011                         2004–2025
                 Fund universe                Active EDC (WFICN)                Active EDC (WFICN)
          Regression N (M3/M4)                            77,140                           135,585
      Unique WFICNs / clusters                             1,215                             1,120
                  M3 coef (SE)                   0.0928 (0.0047)                   0.1171 (0.0043)
+10 pp past return → flow (M3)                   +0.928 pp/month                   +1.171 pp/month
                       Flow SD                           5.22 pp           

### 10.6 Caveats

1. **Missing exp_ratio / turn_ratio (20.5%):** M3/M4 sample is 135,585 vs. 170,616 for M1/M2.
   Same pattern as baseline; both controls are retained for methodological consistency.

2. **FSQ aggregation:** The 2003–2025 file lacks a `tna_latest` column. Categorical
   characteristics use the representative share class (lowest `crsp_fundno`); numeric
   characteristics are unweighted means. Minor deviation; negligible impact on results.

3. **WFICN mapping 79% overall:** After the EDC filter, mapped rate ≥95%.
   Unmapped observations are non-equity funds outside MFLinks coverage.

4. **Structural heterogeneity:** The 2003–2025 sample spans GFC (2008–2009), COVID-19
   (2020), and 2022 rate shock. Subperiod analysis is not conducted due to page limits.

5. **Temporal robustness only:** Both samples use CRSP data with the same methodology.
   This is not a cross-source replication.


## 11. Conclusion

### Main Empirical Finding

This analysis provides consistent evidence of a **positive, monotonic, and convex
flow-performance relationship** for active domestic equity (EDC) funds over 1999–2011:

1. **Positive and significant**: Funds with higher past returns attract more investor
   inflows. The coefficient on `past_12m_return` is positive and significant at
   p < 0.001 in all four model specifications. A fund that outperformed by 10
   percentage points over the previous 12 months attracted approximately **+0.93 pp/month**
   more flows than otherwise identical peers (Model 3, month FE).

2. **Monotone**: Average flows increase monotonically from the bottom-quintile (Q1)
   to the top-quintile (Q5) in every specification. This pattern holds in the
   unadjusted means and in all four regression models.

3. **Convex**: The Q4→Q5 increment (+1.33 pp/month) is **2.4× larger** than the
   average Q1–Q4 step (+0.56 pp), confirming that top-performing funds receive
   **disproportionately** higher inflows. This asymmetric pattern is consistent with
   Sirri & Tufano (1998) and Chevalier & Ellison (1997).

4. **Robust**: Replacing the 12-month with a 6-month performance window produces
   nearly identical conclusions. The SD-normalised effect sizes are 0.48 SDs (12m)
   and 0.46 SDs (6m), and the 6m quintile convexity ratio is even stronger (2.9×).

### Caveats

- Portfolio identifier: WFICN (MFLinks) rather than `crsp_portno` — chosen due to
  sparse `crsp_portno` coverage before 2003.
- Effective sample: 1999-04 to 2011-12 (Fund Summary `crsp_obj_cd` available from
  1999-03 only; 12-month rolling window requires 13 months of data).
- Performance measured by raw cumulative returns, not risk-adjusted alpha.
- Fund universe: active domestic small/mid-cap equity (EDC codes) only.
- Flows winsorized at 1st/99th percentiles; lagged TNA ≥ \$10M required.
- OLS with WFICN-clustered standard errors; no fund fixed effects.

In [26]:
# ── File inventory ───────────────────────────────────────────────────────────
import os
inventory = []
for f in sorted(os.listdir(PROC)):
    if f.startswith('final_'):
        sz = os.path.getsize(PROC + f) / 1e6
        inventory.append({'Location': 'data/processed/', 'File': f, 'Size (MB)': f'{sz:.1f}'})
for f in sorted(os.listdir(TABS)):
    if f.startswith('final_'):
        sz = os.path.getsize(TABS + f) / 1e3
        inventory.append({'Location': 'output/tables/', 'File': f, 'Size (KB)': f'{sz:.1f}'})

inv_df = pd.DataFrame(inventory)
print(inv_df.to_string(index=False))

with open(TABS + 'final_notebook_file_inventory.md', 'w') as f:
    f.write('# Files Created by bachelorthesis_final.ipynb\n\n'
            + tbl_to_md(inv_df) + '\n')
print('\nSaved → final_notebook_file_inventory.md')

       Location                                                 File Size (MB) Size (KB)
data/processed/                final_step1_wficn_portfolio_month.csv      55.4       NaN
data/processed/                final_step2_clean_wficn_edc_flows.csv      35.2       NaN
data/processed/           final_step3_regression_ready_wficn_edc.csv      31.6       NaN
 output/tables/                        final_analysis_overview_qc.md       NaN      11.2
 output/tables/                          final_analysis_pipeline.csv       NaN       1.1
 output/tables/                     final_notebook_file_inventory.md       NaN       2.8
 output/tables/            final_step1_wficn_mapping_diagnostics.csv       NaN       0.9
 output/tables/             final_step1_wficn_mapping_diagnostics.md       NaN       2.2
 output/tables/                           final_step2_exclusions.csv       NaN       0.2
 output/tables/                            final_step2_exclusions.md       NaN       0.3
 output/tables/      